# Регресія тривалості ДТП (`Log_Duration`): DecisionTree vs RandomForest (Spark MLlib)

Дані — `data/processed/regression/` з `regression_data_preparation.ipynb`, ціль `Log_Duration = log1p(Duration_Minutes)`. Навчання, підбір параметрів, метрики й гістограми для графіків виконуються в Spark.

## 0. Налаштування

Параметри запуску задаються змінними середовища: `TRAIN_FRACTION`, `SPARK_PARALLELISM`, `SPARK_DRIVER_MEMORY`, `SHAP_ROWS`. Планувальник FAIR дозволяє паралельним потокам ділити ресурси Spark. Spark UI — на порту 4041, щоб ноутбук міг працювати одночасно з класифікацією.

In [ ]:
import os
import sys
import json
import time
import random
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "main.py").exists() else Path.cwd().parent
os.chdir(PROJECT_ROOT)
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap
from matplotlib.colors import LogNorm
from IPython.display import display

from pyspark import StorageLevel
from pyspark.sql import SparkSession, Window, functions as F
from pyspark.ml.feature import VectorAssembler, Bucketizer
from pyspark.ml.functions import vector_to_array
from pyspark.ml.regression import DecisionTreeRegressor, RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

TASK = "regression"
SEED = 42
TRAIN_FRACTION = float(os.environ.get("TRAIN_FRACTION", "1"))   # частка train для навчання на етапі 3 (1 — весь train)
PARALLELISM = int(os.environ.get("SPARK_PARALLELISM", "2"))      # скільки моделей / фолдів навчається одночасно
FIT_PARALLELISM = int(os.environ.get("SPARK_FIT_PARALLELISM", "1"))   # скільки моделей одночасно вчиться на повному train
SHAP_ROWS = int(os.environ.get("SHAP_ROWS", "3000"))
TUNE_ROWS = 80_000
N_FOLDS = 5
N_CANDIDATES = 15
N_STRATA = 10            # децилі цілі для стратифікації підвибірок
PLOT_BINS = 120          # бінів на вісь у 2D-гістограмах (Actual vs Predicted, Residuals)
DATA_DIR = PROJECT_ROOT / "data" / "processed" / TASK
MODELS_DIR = PROJECT_ROOT / "data" / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
SPARK_TMP = Path(os.environ.get("SPARK_LOCAL_DIR", str(PROJECT_ROOT / "spark-tmp")))

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 220)
pd.set_option("display.max_rows", 300)

spark = (
    SparkSession.builder
    .appName(f"US-Accidents-{TASK}-model-yv")
    .master(os.environ.get("SPARK_MASTER", "local[*]"))
    .config("spark.driver.memory", os.environ.get("SPARK_DRIVER_MEMORY", "16g"))
    .config("spark.driver.host", "localhost")
    .config("spark.ui.port", "4041")
    .config("spark.scheduler.mode", "FAIR")
    .config("spark.sql.shuffle.partitions", "48")
    .config("spark.local.dir", str(SPARK_TMP))
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")
    .config("spark.driver.maxResultSize", "4g")
    .config("spark.ui.showConsoleProgress", "false")
    .config("spark.sql.debug.maxToStringFields", "1000")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
spark.sparkContext.setCheckpointDir(str(SPARK_TMP / "checkpoints" / TASK))
print(f"Spark {spark.version}, master {spark.sparkContext.master}, parallelism {PARALLELISM} (fit {FIT_PARALLELISM}), "
      f"train fraction {TRAIN_FRACTION}, UI {spark.sparkContext.uiWebUrl}")

Допоміжні функції: паралельний запуск Spark-задач у потоках, усі метрики за один прохід Spark (лог-шкала й хвилини), 1D / 2D гістограми в Spark для графіків.

In [ ]:
def run_parallel(fn, keys, workers=None):
    # fn(key) для кожного ключа в workers (за замовчуванням PARALLELISM) потоках; кожен потік відправляє власні Spark-задачі
    keys = list(keys)
    with ThreadPoolExecutor(max_workers=workers or PARALLELISM) as pool:
        return dict(zip(keys, pool.map(fn, keys)))


def evaluate(pred):
    p = pred.select(F.col("label").alias("y_log"), F.col("prediction").alias("p_log"),
                    F.col("minutes").alias("y_min"), F.expm1("prediction").alias("p_min"))
    aggs = []
    for s in ("log", "min"):
        err = F.col(f"y_{s}") - F.col(f"p_{s}")
        aggs += [F.avg(err * err).alias(f"mse_{s}"), F.avg(F.abs(err)).alias(f"mae_{s}"),
                 F.avg(F.abs(err / F.col(f"y_{s}"))).alias(f"mape_{s}"),
                 F.percentile_approx(F.abs(err), 0.5, 10_000).alias(f"medae_{s}"),
                 F.var_pop(f"y_{s}").alias(f"var_{s}")]
    r = p.agg(*aggs).first()
    out = {}
    for s, unit in (("log", "log"), ("min", "minutes")):
        out.update({f"R² ({unit})": 1 - r[f"mse_{s}"] / r[f"var_{s}"], f"MAE ({unit})": r[f"mae_{s}"],
                    f"MSE ({unit})": r[f"mse_{s}"], f"RMSE ({unit})": float(np.sqrt(r[f"mse_{s}"])),
                    f"MAPE % ({unit})": 100 * r[f"mape_{s}"], f"MedAE ({unit})": r[f"medae_{s}"]})
    return out


def with_residuals(pred):
    return pred.select(F.col("label").alias("actual"), F.col("prediction").alias("predicted"),
                       (F.col("label") - F.col("prediction")).alias("residual"))


def bin_index(col, lo, hi, bins):
    return F.floor((F.col(col) - lo) / (hi - lo) * bins).cast("int")


def hist2d(df, x, y, x_range, y_range, bins=PLOT_BINS):
    (x0, x1), (y0, y1) = x_range, y_range
    h = (df.filter(F.col(x).between(x0, x1) & F.col(y).between(y0, y1))
         .select(F.least(bin_index(x, x0, x1, bins), F.lit(bins - 1)).alias("bx"),
                 F.least(bin_index(y, y0, y1, bins), F.lit(bins - 1)).alias("by"))
         .groupBy("bx", "by").count().toPandas())
    grid = np.zeros((bins, bins))
    grid[h["by"].to_numpy(), h["bx"].to_numpy()] = h["count"].to_numpy()
    return grid


def hist1d(df, col, col_range, bins=PLOT_BINS):
    lo, hi = col_range
    h = (df.filter(F.col(col).between(lo, hi))
         .select(F.least(bin_index(col, lo, hi, bins), F.lit(bins - 1)).alias("b"))
         .groupBy("b").count().toPandas())
    counts = np.zeros(bins)
    counts[h["b"].to_numpy()] = h["count"].to_numpy()
    return counts


def plot_hist2d(grid, x_range, y_range, ax, title, xlabel, ylabel):
    im = ax.imshow(np.ma.masked_equal(grid, 0), origin="lower", aspect="auto", cmap="viridis", norm=LogNorm(),
                   extent=[*x_range, *y_range])
    ax.set(title=title, xlabel=xlabel, ylabel=ylabel)
    plt.colorbar(im, ax=ax, label="records")


def plot_actual_vs_predicted(df, ax, title):
    plot_hist2d(hist2d(df, "actual", "predicted", Y_RANGE, Y_RANGE), Y_RANGE, Y_RANGE, ax, title,
                "Actual log1p(duration)", "Predicted log1p(duration)")
    ax.plot(Y_RANGE, Y_RANGE, "r--", lw=1)


def plot_residuals(df, ax, title):
    plot_hist2d(hist2d(df, "predicted", "residual", Y_RANGE, RES_RANGE), Y_RANGE, RES_RANGE, ax, title,
                "Predicted log1p(duration)", "Residual (actual − predicted)")
    ax.axhline(0, color="red", ls="--", lw=1)

## Етап 1. Завантаження та експрес-перевірка даних

Ознаки перейменовуються на `f0…fN`: у назвах є крапки й дужки, які Spark сприймає як доступ до вкладених полів. Оригінальні назви — у `FEATURES`.

In [ ]:
meta = json.loads((DATA_DIR / "metadata.json").read_text(encoding="utf-8"))
FEATURES = meta["feature_names"]
SAFE = [f"f{i}" for i in range(len(FEATURES))]
ORIGINAL_NAME = dict(zip(SAFE, FEATURES))


def load(split):
    df = spark.read.parquet(str(DATA_DIR / split))
    return df.select("ID", F.col("Log_Duration").cast("double"), F.col("Duration_Minutes").cast("double"),
                     *[F.col(f"`{c}`").alias(s) for c, s in zip(FEATURES, SAFE)])


raw = {split: load(split) for split in ("train", "valid", "test")}
if TRAIN_FRACTION < 1:
    raw["train"] = raw["train"].sample(fraction=TRAIN_FRACTION, seed=SEED)

for split, df in raw.items():
    print(f"===== {split}: head =====")
    display(df.limit(5).toPandas().rename(columns=ORIGINAL_NAME))

Аналог `.info()`: тип і кількість пропусків кожної колонки у трьох вибірках.

In [ ]:
info = pd.DataFrame({"dtype": dict(raw["train"].dtypes)})
for split, df in raw.items():
    row = df.select(F.count(F.lit(1)).alias("_rows"),
                    *[F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]).first().asDict()
    print(f"{split}: {row.pop('_rows'):,} rows x {len(df.columns)} columns")
    info[f"nulls {split}"] = pd.Series(row)
info.index = [ORIGINAL_NAME.get(c, c) for c in info.index]
info

Аналог `.describe()` (count, mean, stddev, min, max) — Spark `describe` по кожній вибірці; окремо — квантилі цілі.

In [ ]:
for split, df in raw.items():
    desc = df.drop("ID").describe().toPandas().set_index("summary").T.astype(float)
    desc.index = [ORIGINAL_NAME.get(c, c) for c in desc.index]
    print(f"===== {split}: describe =====")
    display(desc.round(4))

In [ ]:
QS = [0.01, 0.25, 0.5, 0.75, 0.99]
target_q = {(split, col): dict(zip([f"q{int(q * 100)}" for q in QS], df.approxQuantile(col, QS, 0.001)))
            for split, df in raw.items() for col in ("Log_Duration", "Duration_Minutes")}
pd.DataFrame(target_q).T.round(3)

Ознаки збираються у вектор `features`. Децилі цілі з train (`Bucketizer`) задають страти для підвибірок. Вибірки кешуються.

In [ ]:
edges = sorted(set(raw["train"].approxQuantile("Log_Duration", [i / N_STRATA for i in range(1, N_STRATA)], 0.001)))
bucketizer = Bucketizer(splits=[-float("inf"), *edges, float("inf")], inputCol="label", outputCol="stratum")
STRATA = list(range(len(edges) + 1))

assembler = VectorAssembler(inputCols=SAFE, outputCol="features")
data = {split: bucketizer.transform(assembler.transform(df).select(
            "ID", F.col("Log_Duration").alias("label"), F.col("Duration_Minutes").alias("minutes"), "features"))
        for split, df in raw.items()}
data = {split: df.persist(StorageLevel.MEMORY_AND_DISK) for split, df in data.items()}
N_ROWS = run_parallel(lambda split: data[split].count(), data)
print("Cached rows:", {split: f"{n:,}" for split, n in N_ROWS.items()})
print("Decile edges of Log_Duration (train):", np.round(edges, 3).tolist())

## Етап 2. Підбір гіперпараметрів

Підбір виконується на підвибірці ~80 тис. записів train (~1.7%), стратифікованій за децилями цілі: CV на 5 млн рядків тривала б години й ризикувала OOM, а розподіл цілі й зв'язки з ознаками зберігаються. Фолди CV теж стратифіковані за децилями.

In [ ]:
tune_frac = min(1.0, TUNE_ROWS / N_ROWS["train"])
fold_window = Window.partitionBy("stratum").orderBy("_r")
tune = (data["train"].sampleBy("stratum", {float(s): tune_frac for s in STRATA}, seed=SEED)
        .withColumn("_r", F.rand(SEED))
        .withColumn("fold", (F.row_number().over(fold_window) % N_FOLDS).cast("int"))
        .drop("_r")
        .persist(StorageLevel.MEMORY_AND_DISK))
n_tune = tune.count()
print(f"Tuning sample: {n_tune:,} rows ({100 * n_tune / N_ROWS['train']:.2f}% of train)")
pd.DataFrame({name: df.select(F.mean("label").alias("mean"), F.stddev("label").alias("std"),
                              F.percentile_approx("label", [0.1, 0.5, 0.9], 10_000).alias("q")).first().asDict()
              for name, df in (("train", data["train"]), ("sample", tune))}).T

Сімейства й простори пошуку. З повної сітки випадково обирається `N_CANDIDATES` комбінацій (аналог `RandomizedSearchCV`), метрика — RMSE у лог-шкалі на 5-fold CV.

In [ ]:
COMMON = {"seed": SEED, "cacheNodeIds": True, "checkpointInterval": 10}
FAMILIES = {"DecisionTree": DecisionTreeRegressor(**COMMON), "RandomForest": RandomForestRegressor(**COMMON)}
dt, rf = FAMILIES["DecisionTree"], FAMILIES["RandomForest"]
SPACES = {
    "DecisionTree": (ParamGridBuilder()
                     .addGrid(dt.maxDepth, [5, 8, 10, 12, 15, 20])
                     .addGrid(dt.minInstancesPerNode, [1, 10, 50, 200])
                     .addGrid(dt.minInfoGain, [0.0, 1e-4, 1e-3])
                     .addGrid(dt.maxBins, [32, 64, 128])
                     .build()),
    "RandomForest": (ParamGridBuilder()
                     .addGrid(rf.numTrees, [50, 100, 150])
                     .addGrid(rf.maxDepth, [8, 10, 12, 15])
                     .addGrid(rf.featureSubsetStrategy, ["onethird", "sqrt", "0.5"])
                     .addGrid(rf.subsamplingRate, [0.5, 0.8, 1.0])
                     .addGrid(rf.minInstancesPerNode, [1, 10, 50])
                     .build()),
}
CANDIDATES = {fam: random.Random(SEED).sample(grid, min(N_CANDIDATES, len(grid))) for fam, grid in SPACES.items()}
print({fam: f"{len(CANDIDATES[fam])} of {len(grid)} combinations" for fam, grid in SPACES.items()})

In [ ]:
def search(fam):
    cv = CrossValidator(estimator=FAMILIES[fam], estimatorParamMaps=CANDIDATES[fam],
                        evaluator=RegressionEvaluator(metricName="rmse"),
                        numFolds=N_FOLDS, foldCol="fold", parallelism=PARALLELISM, seed=SEED)
    t0 = time.time()
    cv_model = cv.fit(tune)
    table = pd.DataFrame([{p.name: v for p, v in pm.items()} for pm in CANDIDATES[fam]])
    table["RMSE log (CV)"] = cv_model.avgMetrics
    table["std"] = cv_model.stdMetrics
    best = CANDIDATES[fam][int(np.argmin(cv_model.avgMetrics))]
    return best, table.sort_values("RMSE log (CV)"), time.time() - t0


best_params, search_tables = {}, {}
for fam in FAMILIES:
    best_params[fam], search_tables[fam], sec = search(fam)
    print(f"{fam}: {sec:.0f} s, best CV RMSE log {search_tables[fam]['RMSE log (CV)'].iloc[0]:.4f}, "
          f"best params {({p.name: v for p, v in best_params[fam].items()})}")

In [ ]:
for fam, table in search_tables.items():
    print(f"===== {fam}: top 5 of {len(table)} =====")
    display(table.head(5).round(4))

## Етап 3. Навчання та первинне оцінювання моделей

На повному train навчаються 2 варіанти кожного сімейства: base і tuned. Моделі на повному train навчаються по одній (`SPARK_FIT_PARALLELISM`, за замовчуванням 1): дві паралельні RandomForest на 4.8 млн рядків не вміщуються в 10 ГБ пам'яті. Метрики — у лог-шкалі та в хвилинах (`expm1` прогнозу).

In [ ]:
VARIANTS = {f"{fam} {kind}": est.copy(dict(best_params[fam]) if kind == "tuned" else {})
            for fam, est in FAMILIES.items() for kind in ("base", "tuned")}


def fit(name):
    t0 = time.time()
    return VARIANTS[name].fit(data["train"]), time.time() - t0


fitted = run_parallel(fit, VARIANTS, workers=FIT_PARALLELISM)
models = {name: model for name, (model, _) in fitted.items()}
pd.Series({name: round(sec) for name, (_, sec) in fitted.items()}, name="fit seconds").to_frame()

In [ ]:
def metrics_table(split, model_map):
    table = pd.DataFrame(run_parallel(lambda name: evaluate(model_map[name].transform(data[split])), model_map)).T
    table.index = pd.MultiIndex.from_tuples([(name.split()[0], name) for name in table.index], names=["family", "model"])
    return table


results_train = metrics_table("train", models)
results_train.round(4)

## Етап 4. Оцінювання на валідації

In [ ]:
results_valid = metrics_table("valid", models)
LEADERS = {fam: results_valid.loc[fam, "RMSE (log)"].idxmin() for fam in FAMILIES}
print("Leaders by valid RMSE (log):", LEADERS)
results_valid.round(4)

Графіки лідерів на train і valid — 2D-гістограми всіх записів, пораховані в Spark (замість scatter мільйонів точок). Межі осей — квантилі 0.1% / 99.9%.

In [ ]:
leader_preds = {(fam, split): with_residuals(models[name].transform(data[split])).persist(StorageLevel.MEMORY_AND_DISK)
                for fam, name in LEADERS.items() for split in ("train", "valid", "test")}
Y_RANGE = tuple(data["train"].approxQuantile("label", [0.001, 0.999], 0.0001))
res_lim = max(abs(q) for key in leader_preds if key[1] == "valid"
              for q in leader_preds[key].approxQuantile("residual", [0.001, 0.999], 0.0001))
RES_RANGE = (-res_lim, res_lim)

fig, axes = plt.subplots(2, 2, figsize=(14, 11))
for i, (fam, name) in enumerate(LEADERS.items()):
    for j, split in enumerate(("train", "valid")):
        plot_actual_vs_predicted(leader_preds[(fam, split)], axes[i, j], f"{name} — {split}")
fig.suptitle("Actual vs Predicted: train vs valid", fontsize=14)
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(2, 2, figsize=(14, 11))
for i, (fam, name) in enumerate(LEADERS.items()):
    for j, split in enumerate(("train", "valid")):
        plot_residuals(leader_preds[(fam, split)], axes[i, j], f"{name} — {split}")
fig.suptitle("Residuals vs Predicted: train vs valid", fontsize=14)
plt.tight_layout()
plt.show()

## Етап 5. Фінальне оцінювання на тестовій вибірці

In [ ]:
results_test = pd.DataFrame(run_parallel(lambda fam: evaluate(models[LEADERS[fam]].transform(data["test"])), LEADERS)).T
results_test.index = [LEADERS[fam] for fam in results_test.index]
results_test.T.round(4)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
for ax, (fam, name) in zip(axes, LEADERS.items()):
    plot_actual_vs_predicted(leader_preds[(fam, "test")], ax, f"{name} — test")
fig.suptitle("Actual vs Predicted: test", fontsize=14)
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
for ax, (fam, name) in zip(axes, LEADERS.items()):
    plot_residuals(leader_preds[(fam, "test")], ax, f"{name} — test")
fig.suptitle("Residuals vs Predicted: test", fontsize=14)
plt.tight_layout()
plt.show()

edges_res = np.linspace(*RES_RANGE, PLOT_BINS + 1)
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for ax, (fam, name) in zip(axes, LEADERS.items()):
    counts = hist1d(leader_preds[(fam, "test")], "residual", RES_RANGE)
    ax.bar(edges_res[:-1], counts, width=np.diff(edges_res), align="edge", color="steelblue")
    ax.axvline(0, color="red", ls="--", lw=1)
    ax.set(title=f"{name} — test", xlabel="Residual (actual − predicted), log scale", ylabel="Records")
fig.suptitle("Distribution of residuals: test", fontsize=14)
plt.tight_layout()
plt.show()

## Етап 6. SHAP-аналіз та інтерпретованість

Семпл ~3 000 записів test, стратифікований за децилями цілі. `shap.TreeExplainer` читає дерева Spark-моделі напряму й пояснює прогноз `log1p(тривалості)`.

In [ ]:
shap_frac = min(1.0, SHAP_ROWS / N_ROWS["test"])
shap_rows = (data["test"].sampleBy("stratum", {float(s): shap_frac for s in STRATA}, seed=SEED)
             .select(vector_to_array("features").alias("x")).toPandas())
X_shap = pd.DataFrame(np.stack(shap_rows["x"].to_numpy()), columns=FEATURES)
print(f"SHAP sample: {len(X_shap):,} rows")

shap_values = {}
for fam, name in LEADERS.items():
    t0 = time.time()
    shap_values[fam] = shap.TreeExplainer(models[name])(X_shap)
    print(f"{name}: {time.time() - t0:.0f} s, SHAP values {shap_values[fam].values.shape}")

In [ ]:
for fam, sv in shap_values.items():
    shap.plots.bar(sv, max_display=20, show=False)
    plt.title(f"{LEADERS[fam]}: mean |SHAP| (top 20 features)")
    plt.show()
    shap.plots.beeswarm(sv, max_display=20, show=False)
    plt.title(f"{LEADERS[fam]}: SHAP beeswarm")
    plt.show()

## Збереження лідерів

In [ ]:
for fam, name in LEADERS.items():
    path = MODELS_DIR / f"{TASK}_yv_{fam}"
    models[name].write().overwrite().save(str(path))
    info = {"variant": name, "params": {p.name: v for p, v in models[name].extractParamMap().items()},
            "features": FEATURES, "target": "Log_Duration", "inverse": "expm1"}
    path.with_suffix(".json").write_text(json.dumps(info, indent=2, default=str), encoding="utf-8")
    print(f"Saved {name} -> {path}")

In [ ]:
for df in [*data.values(), *leader_preds.values(), tune]:
    df.unpersist()
spark.stop()